In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Đường dẫn tương đối: phải chạy voila từ thư mục gốc dự án
MODEL_PATH: Path = Path("artifacts/linear_regression.joblib")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

artifact = joblib.load(MODEL_PATH)                     # chỉ nạp mô hình, KHÔNG huấn luyện lại
model = artifact["model"]                              # Pipeline đã học xong
feature_names: list[str] = artifact["feature_names"]   # thứ tự cột lúc huấn luyện

In [ ]:
# ===== 1. CSS: toàn bộ giao diện nằm ở đây =====
CSS: str = """
<style>
  .sp-header {
    max-width: 680px; margin: 16px auto 8px; padding: 28px 24px; border-radius: 18px;
    background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 100%); color: #fff;
    font-family: 'Segoe UI', Arial, sans-serif; text-align: center;
    box-shadow: 0 8px 24px rgba(79, 70, 229, .25);
  }
  .sp-header h1 { margin: 0 0 6px; font-size: 26px; font-weight: 700; }
  .sp-header p  { margin: 0; font-size: 14px; opacity: .9; }
  .sp-note {
    max-width: 680px; margin: 0 auto 8px; padding: 8px 14px; font-size: 12.5px;
    font-family: 'Segoe UI', Arial, sans-serif; color: #92400e;
    background: #fef3c7; border-radius: 10px; text-align: center;
  }
  .sp-result {
    max-width: 680px; margin: 8px auto; padding: 22px 24px; border-radius: 18px;
    font-family: 'Segoe UI', Arial, sans-serif; text-align: center;
    background: #fff; border: 2px solid var(--c); box-shadow: 0 6px 20px rgba(0, 0, 0, .08);
  }
  .sp-result .label { font-size: 13px; color: #6b7280; text-transform: uppercase; letter-spacing: .08em; }
  .sp-result .score { font-size: 56px; font-weight: 800; line-height: 1.1; color: var(--c); }
  .sp-result .score small { font-size: 22px; color: #9ca3af; font-weight: 600; }
  .sp-result .badge {
    display: inline-block; margin-top: 6px; padding: 4px 16px; border-radius: 999px;
    font-size: 14px; font-weight: 600; color: #fff; background: var(--c);
  }
  .sp-bar { height: 10px; margin-top: 16px; background: #e5e7eb; border-radius: 999px; overflow: hidden; }
  .sp-bar > div { height: 100%; width: var(--w); background: var(--c); border-radius: 999px; }
  .sp-error { color: #b91c1c; font-family: 'Segoe UI', Arial, sans-serif; text-align: center; }
</style>
"""

HEADER: str = """
<div class="sp-header">
  <h1>🎓 Dự đoán điểm thi sinh viên</h1>
  <p>Mô hình hồi quy tuyến tính · kéo thanh trượt rồi bấm <b>Dự đoán</b></p>
</div>
<div class="sp-note">Chỉ phục vụ giảng dạy, dùng dữ liệu tổng hợp. Không dùng để đánh giá sinh viên thật.</div>
"""

# ===== 2. Các ô nhập liệu =====
STYLE: dict[str, str] = {"description_width": "170px", "handle_color": "#4f46e5"}
FULL: widgets.Layout = widgets.Layout(width="100%")

DEFAULTS: dict[str, float] = {
    "hours": 6, "attendance": 80, "assignments": 7, "previous": 6.5
}

hours = widgets.FloatSlider(value=DEFAULTS["hours"], min=0, max=20, step=.5,
                            description="⏱ Giờ học / tuần", style=STYLE, layout=FULL,
                            continuous_update=False, readout_format=".1f")
attendance = widgets.FloatSlider(value=DEFAULTS["attendance"], min=50, max=100, step=1,
                                 description="📅 Chuyên cần (%)", style=STYLE, layout=FULL,
                                 continuous_update=False, readout_format=".0f")
assignments = widgets.IntSlider(value=int(DEFAULTS["assignments"]), min=0, max=10, step=1,
                                description="📝 Số bài tập nộp", style=STYLE, layout=FULL,
                                continuous_update=False)
previous = widgets.FloatSlider(value=DEFAULTS["previous"], min=0, max=10, step=.1,
                               description="📊 Điểm trước đó", style=STYLE, layout=FULL,
                               continuous_update=False, readout_format=".1f")

predict_button = widgets.Button(description="Dự đoán", icon="calculator",
                                button_style="primary",
                                layout=widgets.Layout(width="180px", height="42px"))
reset_button = widgets.Button(description="Đặt lại", icon="refresh",
                              layout=widgets.Layout(width="140px", height="42px"))
output = widgets.Output()  # vùng hiển thị kết quả


# ===== 3. Hàm xử lý =====
def input_frame() -> pd.DataFrame:
    """Gom giá trị các thanh trượt thành 1 dòng DataFrame đúng thứ tự cột."""
    values = {
        "hours_studied": hours.value,
        "attendance": attendance.value,
        "assignments_completed": assignments.value,
        "previous_score": previous.value,
    }
    return pd.DataFrame(
        [[values[name] for name in feature_names]], columns=feature_names
    )


def grade(score: float) -> tuple[str, str]:
    """Trả về (xếp loại, màu) theo điểm."""
    if score >= 8:
        return "Giỏi", "#16a34a"
    if score >= 6.5:
        return "Khá", "#2563eb"
    if score >= 5:
        return "Trung bình", "#d97706"
    return "Cần cố gắng", "#dc2626"


def result_card(score: float) -> widgets.HTML:
    """Tạo thẻ kết quả: điểm lớn, xếp loại và thanh tiến độ."""
    text, color = grade(score)
    html = (
        f'<div class="sp-result" style="--c:{color}; --w:{score * 10:.0f}%">'
        f'<div class="label">Điểm thi dự đoán</div>'
        f'<div class="score">{score:.2f}<small> / 10</small></div>'
        f'<div class="badge">{text}</div>'
        f'<div class="sp-bar"><div></div></div></div>'
    )
    return widgets.HTML(html)


def on_predict(_: object) -> None:
    """Chạy mỗi khi bấm nút Dự đoán."""
    predict_button.disabled = True           # khóa nút, tránh bấm liên tục
    with output:
        output.clear_output()                # xóa kết quả cũ
        try:
            raw = float(model.predict(input_frame())[0])
            score = float(np.clip(raw, 0, 10))   # cắt về thang 0-10
            display(result_card(score))
        except Exception:
            # Không in traceback chi tiết cho người dùng
            display(widgets.HTML(
                '<div class="sp-error">Không thể dự đoán. Hãy kiểm tra dữ liệu và artifact.</div>'
            ))
        finally:
            predict_button.disabled = False  # luôn mở khóa nút lại


def on_reset(_: object) -> None:
    """Đưa các thanh trượt về giá trị mặc định và xóa kết quả."""
    hours.value = DEFAULTS["hours"]
    attendance.value = DEFAULTS["attendance"]
    assignments.value = int(DEFAULTS["assignments"])
    previous.value = DEFAULTS["previous"]
    output.clear_output()


predict_button.on_click(on_predict)
reset_button.on_click(on_reset)

# ===== 4. Ráp giao diện =====
form = widgets.VBox(
    [hours, attendance, assignments, previous],
    layout=widgets.Layout(width="100%", padding="20px 24px", gap="6px",
                          border="1px solid #e5e7eb", border_radius="16px"),
)
buttons = widgets.HBox([predict_button, reset_button],
                       layout=widgets.Layout(justify_content="center", gap="12px",
                                             margin="14px 0"))

page = widgets.VBox(
    [widgets.HTML(CSS + HEADER), form, buttons, output],
    layout=widgets.Layout(max_width="680px", margin="0 auto", padding="0 12px"),
)
display(page)

In [ ]:
s